# Introduction 🐥🐥

This lesson mixes short explanations with small demos 🔬. You don't need to understand every line of code yet: the goal is to look at the output and read the comment below each demo. The next cell loads the libraries used in the demos.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor

import plotly.io as pio
import plotly.graph_objects as go
pio.templates["course"] = go.layout.Template(layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"])
pio.templates.default = "plotly_white+course"

## What is machine learning?

**Machine learning** is a multidisciplinary science that aims to enable machines to solve complex problems without explicitly coding the decision-making process. For example, predicting the conversion rate on a website or the price of real estate are problems that can be solved by optimizing machine learning models.

ai

:::tip Intuition
In classic programming, a human writes the rules ("if the surface is above 50 m², then..."), and the computer applies them. In machine learning, we give the computer **examples** and an algorithm finds the rules by itself. The more complex the rules (recognizing a cat in a picture, estimating a price from 20 characteristics), the more useful this becomes.
:::

## What is supervised machine learning?

**Supervised machine learning** is a branch of this discipline that solves problems for which we already have solved examples. For instance, suppose we have collected data on a sample of housing units in San Francisco: their location, various characteristics and their rent. Our problem is to estimate the rent of a dwelling in the same city that is not in our database yet. We build a model from our data that estimates the rent based on the characteristics of the dwellings, then we apply this model to the new dwelling to estimate its rent. This is a typical supervised learning problem: when the model was built, the values of the **variable** to be estimated were known. We call this variable the **target variable**.

:::tip Intuition
Think of a student who prepares an exam with corrected exercises. The corrected exercises are the **solved examples** (the data where the target is known). Doing well on exercises you have already seen with their answers proves nothing: what matters is the exam, with **new** questions. In machine learning, the ability to perform well on new data is called **generalization**, and it is what we always try to measure.
:::

#### See it with data 🔬: learning from solved examples

Let's build a small rental dataset in the spirit of the example above. Each row is a dwelling with its surface (in m²), its distance to the city center (in km) and its monthly rent (in dollars). Because we simulate the data ourselves, we know the true rule behind the rents: `rent = 500 + 25 × surface - 60 × distance + noise`.

We hide 20% of the dwellings (the **test set**), let a linear regression model learn from the other 80% (the **training set**), and then ask it to estimate the rent of the hidden dwellings.

In [ ]:
rng = np.random.default_rng(0)
n = 500
surface = rng.uniform(15, 120, n).round()
distance = rng.uniform(0, 15, n).round(1)
noise = rng.normal(0, 200, n)                      # what the characteristics can't explain
rent = 500 + 25 * surface - 60 * distance + noise
homes = pd.DataFrame({"surface": surface, "distance": distance, "rent": rent.round()})

X, y = homes[["surface", "distance"]], homes["rent"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print("Training set:", X_train.shape, " Test set:", X_test.shape)
homes.head()

Training set: (400, 2)  Test set: (100, 2)


,surface,distance,rent
0,82.0,1.2,2495.0
1,43.0,13.1,968.0
2,19.0,14.1,570.0
3,17.0,3.9,837.0
4,100.0,0.2,2707.0


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)            # learning: only the training set is used

new_homes = X_test.head(5).copy()      # dwellings the model has never seen
new_homes["true rent"] = y_test.head(5)
new_homes["predicted rent"] = model.predict(X_test.head(5)).round()
new_homes

,surface,distance,true rent,predicted rent
90,112.0,13.9,2391.0,2456.0
254,66.0,2.0,2256.0,2042.0
283,65.0,3.6,1625.0,1921.0
445,103.0,5.4,2558.0,2746.0
461,20.0,8.0,577.0,551.0


In [ ]:
fig = px.scatter(homes, x="surface", y="rent", color="distance",
                 title="Simulated rents: bigger dwellings cost more, distant ones cost less",
                 labels={"surface": "Surface (m²)", "rent": "Monthly rent ($)", "distance": "Distance (km)"})
fig.show()

The model has never seen the five dwellings above, yet its estimates land within 300 dollars of the true rents (for example, for the 20 m² dwelling 8 km from the center, the true rent is 577 dollars and the model predicts 551 dollars). It learned the general rule from the 400 training examples, and it can apply that rule to new dwellings. This is supervised learning in a nutshell.

#### See it with data 🔬: what "generalize" means

To check that a model generalizes, we compare its score on the training set with its score on the test set. Here we use the R² score (1 is a perfect prediction, 0 is no better than always predicting the average rent; you'll learn it in the linear regression module). We compare our linear model with a very flexible model, a deep decision tree (you'll meet it later), which is able to memorize the training set.

In [ ]:
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)

for name, m in [("Linear regression", model), ("Deep decision tree", tree)]:
    print(f"{name:20s} train R² = {m.score(X_train, y_train):.3f}   test R² = {m.score(X_test, y_test):.3f}")

Linear regression    train R² = 0.939   test R² = 0.940
Deep decision tree   train R² = 1.000   test R² = 0.815


The linear model scores about the same on both sets (train R² = 0.939, test R² = 0.940): what it learned on the training set still works on new data, so it **generalizes**. The deep tree gets a perfect score on the training set (R² = 1.000) because it memorized every training dwelling, noise included, but it drops to R² = 0.815 on the test set. Like a student who learned the answers by heart instead of the method: a perfect score on the training data is not a sign of a good model. This is why we always evaluate models on data they have not seen.

## Regression / Classification: the most common types of supervised machine learning problems
Most supervised machine learning problems can be sorted into two categories:
* Regression: a regression problem is a situation where your target variable is quantitative
* Classification: a classification problem means your target variable is categorical

Any machine learning model can be formally presented in the following way:

$$Y = f(X) + \epsilon$$

or sometimes, in the case of classification (when the target variable is categorical):

$$P(Y) = f(X) + \epsilon$$

where:
* $Y$ is the target variable (usually a vector of $n$ elements, one for each of the $n$ observations);
* $X$ represents the explanatory variables (usually a matrix with $n$ rows, one for each of the $n$ observations in the dataset, and $p$ columns, one for each of the $p$ explanatory variables);
* $f$ is the true function that links the explanatory variables to the target variable. The goal of the data scientist is to train a model that approximates $f$ as closely as possible;
* $\epsilon$ is the noise: an unpredictable element that may come from information we do not observe (outside the scope of our explanatory variables), from the measurement of $Y$, or from other random perturbations.

:::tip Intuition
The formula $Y = f(X) + \epsilon$ says: "the target is a rule applied to the explanatory variables, plus something we can't predict". Two dwellings with exactly the same surface and the same distance to the center can still have different rents (a nicer view, a friendlier landlord...). That part is $\epsilon$: no model, however clever, can predict it from $X$.
:::

#### See it with data 🔬: the noise sets a limit

In our simulated data we know the true $f$ (`500 + 25 × surface - 60 × distance`), so we can measure the error of the **perfect** model and compare it with the error of our learned model on the test set. We use the average size of the errors (the mean absolute error).

In [ ]:
def true_f(X):
    return 500 + 25 * X["surface"] - 60 * X["distance"]

error_perfect = np.mean(np.abs(y_test - true_f(X_test)))
error_model = np.mean(np.abs(y_test - model.predict(X_test)))
print(f"Average error of the true f      : {error_perfect:.0f} $")
print(f"Average error of the learned model: {error_model:.0f} $")

Average error of the true f      : 163 $
Average error of the learned model: 163 $


Both errors are about 163 dollars: our learned model is already as good as the true rule, and even the perfect $f$ misses the rents by 163 dollars on average. This remaining error is the noise $\epsilon$. Data scientists try to get as close as possible to $f$, but they can never go below the noise.

#### See it with data 🔬: regression or classification, same data

The type of problem depends on the **target**, not on the explanatory variables. With the same dwellings, we can ask two questions:
* Regression: "what is the rent?" (a number);
* Classification: "is the rent above 2,000 dollars?" (a category: yes or no).

In [ ]:
y_train_class = (y_train > 2000).map({True: "expensive", False: "affordable"})

regressor = LinearRegression().fit(X_train, y_train)
classifier = LogisticRegression(max_iter=1000).fit(X_train, y_train_class)

answers = X_test.head(4).copy()
answers["regression: rent"] = regressor.predict(X_test.head(4)).round()
answers["classification: class"] = classifier.predict(X_test.head(4))
answers["P(expensive)"] = classifier.predict_proba(X_test.head(4))[:, 1].round(2)   # column 1 = "expensive"
answers

,surface,distance,regression: rent,classification: class,P(expensive)
90,112.0,13.9,2456.0,expensive,0.95
254,66.0,2.0,2042.0,expensive,0.57
283,65.0,3.6,1921.0,affordable,0.34
445,103.0,5.4,2746.0,expensive,1.00


The regression model answers with a number (2,042 dollars for the second dwelling), while the classification model answers with a category ("expensive") and a probability (0.57). Notice that this dwelling is close to the 2,000 dollars limit, so the classifier is unsure (0.57 is close to 0.5), whereas it is very confident for the fourth one (1.00). This probability is the $P(Y)$ of the formula above.

In this module we will explore many different models. They are nothing more than different ways to approximate $f$, and each of them has advantages and shortcomings. At the end of this module, you will understand how they all work, how they are trained on data, and what their pros and cons are.

## Why should you use machine learning? 🤔🤔

Once a problem has been identified, a target variable has been selected, and a number of explanatory variables have been collected, the objectives of supervised machine learning fall into three broad, non-exclusive categories:


* Description: we try to understand the relations between the target variable "Y" and the explanatory variables "X1, ..., Xp", in order to select the most relevant ones or to visualize the behaviors in the observed samples. For example, if Facebook wants to understand which users are most receptive to its ads and why, it may train a model that predicts the probability that a user clicks on an ad, and then look at which variables are the most influential. This may reveal the right levers to get more users to click on ads.
* Explanation: when you have expert knowledge of the data used in your machine learning problem (in economics or biology, for example), the objective is to build a test that lets you verify or confirm theoretical results in practical situations.
* Prediction: here we focus on the quality of the predictions. We do not necessarily try to understand the observed population as well as possible; we want a model that produces reliable predictions for future observations. This is probably the situation you have encountered the most, without knowing it. When you want a prediction model inside an app or a website, to make personalized recommendations or to block fraudulent payments automatically without human intervention, you need a model that makes the best possible predictions, regardless of its complexity or interpretability.

machine learning

#### See it with data 🔬: description and prediction with the same model

The same trained model can serve different objectives. For **description**, we look inside the model: the coefficients of the linear regression tell us how the rent changes with each variable. For **prediction**, we only care about how close the estimates are on new data.

In [ ]:
print("Description: what did the model learn?")
for name, coef in zip(X_train.columns, model.coef_):
    print(f"  +1 unit of {name:8s} -> rent changes by {coef:+.1f} $")
print(f"  intercept: {model.intercept_:.0f} $")

print("\nPrediction: how good is it on new dwellings?")
print(f"  test R² = {model.score(X_test, y_test):.3f}")

Description: what did the model learn?
  +1 unit of surface  -> rent changes by +24.6 $
  +1 unit of distance -> rent changes by -60.1 $
  intercept: 541 $

Prediction: how good is it on new dwellings?
  test R² = 0.940


The model recovered the rule we used to simulate the data: +24.6 dollars per extra m² (true value: 25) and -60.1 dollars per km from the center (true value: -60). This is **description**: the model tells us something about how rents work. For **prediction**, the test R² of 0.940 tells us how reliable the estimates are on new dwellings. On real data we don't know the true rule, but the logic is the same.